# Starting off
Minimal examples to test my understanding of the LLM and the tools.

In [1]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch
import pandas as pd

df = pd.read_csv('./knowledge/clean_youtube_watch_log.csv')
title = df['video_title'].iloc[0]
thumbnail = Image.open(f'./knowledge/thumbnails/{df["video_id"].iloc[0]}.jpg')

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [2]:
def embed_text(text: str):
    '''Embed a text using CLIP'''
    inputs = proc(text=[text], return_tensors="pt", truncation=True, padding=True)  # WARNING! Truncation may lead to loss of information for long texts
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True) # Normalize 

def embed_image(image: Image.Image):
    '''Embed an image using CLIP'''
    inputs = proc(images=image, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True) # Normalize

def similarity(a: torch.Tensor, b: torch.Tensor):
    '''Calculate cosine similarity between two embeddings'''
    return (a @ b.T).item()

In [3]:
# text / text
sim = similarity(embed_text("cat"), embed_text("dog"))
print(f"Similarity between 'cat' and 'dog': {sim:.4f}")

# text / image
img = Image.open("./knowledge/thumbnails/-n8GYm35FeA.jpg")
sim = similarity(embed_text("car"), embed_image(img))
print(f"Similarity between 'car' and thumbnail: {sim:.4f}")

Similarity between 'cat' and 'dog': 0.9122
Similarity between 'car' and thumbnail: 0.2566


# Indexing the embeddings of whole dataset in FAISS

In [ ]:
import faiss
from tqdm.notebook import tqdm

def build_index(texts: list[str], images: list[Image.Image]):
    text_vecs = []
    img_vecs = []

    # Embed all texts and images
    for t, img in tqdm(zip(texts, images), total=len(texts), desc="Embedding Image / Text pairs"):
        text_vecs.append(embed_text(t))
        img_vecs.append(embed_image(img))

    # Convert to numpy arrays
    text_vecs = torch.cat(text_vecs).cpu().numpy()
    img_vecs  = torch.cat(img_vecs).cpu().numpy()

    # Find embedding dimension
    d = text_vecs.shape[1]

    # Createa and populate the indices
    index_text = faiss.IndexFlatIP(d)  # cosine (because normalized)
    index_img  = faiss.IndexFlatIP(d)
    index_text.add(text_vecs)
    index_img.add(img_vecs)

    return index_text, index_img

In [ ]:
texts = df['video_title'].tolist()
images = [Image.open(f'./knowledge/thumbnails/{vid_id}.jpg') for vid_id in df['video_id']]
index_text, index_img = build_index(texts, images)

## Implement similarity search for text and images

In [ ]:
def search_index(query: str, index: faiss.Index, k=5):
    q_vec = embed_text(query).cpu().numpy()
    scores, ids = index.search(q_vec, k)
    return scores[0], ids[0]

In [ ]:
query = "funny cat"

# Search for similar video titles
scores, ids = search_index(query, index_text)
print(f"Top results for query '{query}':")
for score, idx in zip(scores, ids):
    print(f"  - {df['video_title'].iloc[idx]:60} (score: {score:6.4f})")

# Search for similar thumbnails
scores, ids = search_index(query, index_img)
print(f"\nTop thumbnails for query '{query}':")
for score, idx in zip(scores, ids):
    print(f"  - {df['video_title'].iloc[idx]:60} (score: {score:6.4f})")

# Indexing the embeddings of whole dataset in ChromaDB

In [ ]:
import chromadb
from tqdm.notebook import tqdm

def build_index(ids, titles, thumbnails):
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.create_collection(name="youtube_videos")
    
    # Embed all texts and images
    vecs = []
    for t, img in tqdm(zip(titles, thumbnails), total=len(titles), desc="Embedding Image / Text pairs"):
        text_emb = embed_text(t)
        img_emb  = embed_image(img)
        # fusion: 70% text, 30% image
        vecs.append(0.7 * text_emb + 0.3 * img_emb)

    # Convert to numpy array
    vecs = torch.cat(vecs).cpu().numpy()

    # Add embeddings to the collection
    collection.add(
        ids=ids,            # identifier for each entry (e.g., video ID)
        embeddings=vecs,    # list of embeddings (fused text + image)
        documents=titles    # optional: store the original titles as metadata
    )

    return collection

def load_index():
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_collection(name="youtube_videos")
    return collection

In [ ]:
ids = list(dict.fromkeys(df['video_id']))       # ignore duplicates
titles = [df[df['video_id'] == vid]['video_title'].iloc[0] for vid in ids]
images = [Image.open(f'./knowledge/thumbnails/{vid}.jpg') for vid in ids]

# index = build_index(ids, titles, images)
index = load_index()

## Implement similarity search for text and images

In [ ]:
def search_index(query: str, index: chromadb.Collection, k=5): 
    q = embed_text(query).squeeze().tolist()

    res = index.query(
        query_embeddings=[q],
        n_results=k
    )

    return res["ids"][0], res["documents"][0], res["distances"][0]

In [ ]:
query = "funny cat"

# Search for similar titles
ids, titles, distances = search_index(query, index)
print(f"Top results for query '{query}':")
for id, title, distance in zip(ids, titles, distances):
    print(f"  - {title:60} (distance: {distance:6.4f}, id: {id})")

# Building the simplest LangGraph agent

In [23]:
from dataclasses import dataclass
from typing import Annotated, TypedDict
import pandas as pd
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from dataclasses import asdict
import ipywidgets as widgets
from PIL import Image
from IPython.display import display
from IPython.display import clear_output
import html
import ipywidgets as widgets
from IPython.display import display


CSV_PATH = './knowledge/clean_youtube_watch_log.csv'
THUMBNAIL_PATH = './knowledge/thumbnails/'

llm = ChatOllama(
    model="qwen3.5:2b",
    temperature=0
)

## 1. Data types

In [24]:
@dataclass
class WatchItem:
    user_id: str
    video_id: str
    video_title: str
    timestamp: str      # ISO format

    @property
    def thumbnail(self) -> Image.Image:
        return Image.open(f'./knowledge/thumbnails/{self.video_id}.jpg')

## 2. Tools

In [25]:
@tool
def retrieve_session(user_id: str, sorted = True, n: int = None) -> list[WatchItem]:
    """
    Loads a user's watch session, sorted by the most recent first. 
    (Optionally) limited to the top N results.
    """
    df = pd.read_csv(CSV_PATH)
    df = df[df['user_id'] == user_id]   # Filter by user ID

    # Sort and limit
    if sorted:
        df = df.sort_values("watch_date", ascending=False)
    if n is not None:
        df = df.head(n)

    # Map to dataclass
    return df.apply(
        lambda row: asdict(WatchItem(
            user_id=row['user_id'],
            video_id=row['video_id'],
            video_title=row['video_title'],
            timestamp=row['watch_date']
        )), axis=1).tolist()

TOOLS = [retrieve_session]

## 3. Building the agent

In [26]:
# Bind the tools
llm = llm.bind_tools([retrieve_session])

# State definition for LangGraph
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


# ---------- Node 1: LLM agent  ----------
SYSTEM_PROMPT = """
You are a YouTube watch-history analysis agent for rabbit hole prevention.

You help answer questions about a user's history.

Rules:
- If you need the list of videos a user watched, call retrieve_session.
- You can limit the number of videos retrieved if you want, but by default retrieve the whole session.
- When you know the answer, respond directly to the user in one short sentence.
"""

def agent_node(state: AgentState):
    """
    When LLM node is invoked, it receives the System Prompt + the conversation history. 
    After reasoning, it creates a new message with either an answer or a tool call.
    """
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


# ---------- Node 2: Tool router ----------
def route_tools(state: AgentState):
    """
    When router node is invoked, it checks the last message (from LLM node) for tool calls. 
    If there are tool calls, it routes to the tools node, otherwise it ends the graph execution.
    """
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END


# ---------- Graph ----------
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(TOOLS))      # pre-built tool node

# Add edges
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", route_tools, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

graph = builder.compile()

# 4. Testing the agent

In [ ]:
user_id = "25"
input_box = widgets.Text(placeholder="You: ")
output_box = widgets.HTML()


def show(text):
    output_box.value = f"""
    <div style="white-space: pre-wrap; overflow-wrap: anywhere; font-family: monospace;">
        {html.escape(text)}
    </div>
    """


def on_submit(change):
    input_box.disabled = True
    user_msg = input_box.value.strip()

    if user_msg.lower() == "exit":
        return

    prompt = (
        f"The user_id is {user_id}. "
        f"Answer to this question about that user's session: {user_msg}"
    )

    stream = graph.stream(
        {"messages": [HumanMessage(content=prompt)]},
        stream_mode="messages",
        version="v2",
    )

    show("🪨 is 🤔 - please wait")

    output = ""
    for part in stream:
        content = part["data"][0].content if "data" in part and isinstance(part["data"], tuple) and hasattr(part["data"][0], "content") else None
        if content:
            output += part["data"][0].content
            show(output)

    input_box.value = ""
    input_box.disabled = False


input_box.on_submit(on_submit)
display(input_box, output_box)

/var/folders/gq/c0m9vgqx2_vflttb1fnkyw1r0000gn/T/ipykernel_36388/1284370044.py:45: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_submit)


Text(value='', placeholder='You: ')

HTML(value='')

/var/folders/gq/c0m9vgqx2_vflttb1fnkyw1r0000gn/T/ipykernel_36388/1284370044.py:45: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_submit)


Text(value='', placeholder='You: ')

HTML(value='')